# Two-Stage Teaching Assignment Model (PuLP)

This notebook implements a **simple two-stage stochastic teaching assignment model** using **PuLP**.

## Modeling logic
- **Stage 1**: assign all full-time faculty and some planned adjunct teaching to cover the **baseline demand** (`MinGroups`)
- **Stage 2**: use **Monte Carlo scenarios** for random course group numbers between `MinGroups` and `MaxGroups`
- **Recourse**: if realized demand is higher than the stage-1 plan, add **extra adjunct groups**
- **Preference** is treated as a **soft penalty**, not a hard constraint



In [5]:
# If PuLP is not installed, uncomment the next line:
# !pip install pulp


In [6]:
import pandas as pd
import numpy as np
import pulp
from pathlib import Path


In [7]:
# =========================
# 1. Locate and read the Excel file
# =========================
df = pd.read_excel("data_for_the_case (1).xlsx")

print(df.shape)
df.head()


(45, 17)


,Course Code,Credits,Trimester,Number of Groups,Unnamed: 4,Faculty 1,Faculty 2,Faculty 3,Faculty 4,Faculty 5,Faculty 6,Faculty 7,Faculty 8,Faculty 9,Faculty 10,Faculty 11,Faculty 12
0,NaN,NaN,NaN,Min,Max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,T101,3.0,1.0,3,3,0.0,5.0,2.0,0.0,5.0,4.0,3.0,0.0,3.0,5.0,2.0,2.0
2,T102,2.0,1.0,3,3,1.0,5.0,4.0,5.0,5.0,1.0,4.0,0.0,0.0,5.0,0.0,5.0
3,T103,2.0,1.0,2,2,1.0,0.0,4.0,1.0,3.0,0.0,3.0,1.0,3.0,4.0,2.0,5.0
4,T104,2.0,1.0,3,4,2.0,2.0,0.0,0.0,1.0,4.0,0.0,5.0,2.0,1.0,1.0,5.0


In [8]:
# =========================
# 2. Prepare the data
# =========================
# Course rows: Excel rows 2-43 -> pandas rows 1:43
course_df = df.iloc[1:43].copy()

course_df.columns = [
    "Course", "Credits", "Trimester", "MinGroups", "MaxGroups",
    "Faculty 1", "Faculty 2", "Faculty 3", "Faculty 4", "Faculty 5", "Faculty 6",
    "Faculty 7", "Faculty 8", "Faculty 9", "Faculty 10", "Faculty 11", "Faculty 12"
]

faculty_cols = [f"Faculty {i}" for i in range(1, 13)]

# Last two rows are min / max faculty load
min_load_row = df.iloc[43, 5:17].astype(float).values
max_load_row = df.iloc[44, 5:17].astype(float).values

courses = course_df["Course"].tolist()
faculties = faculty_cols

credits = dict(zip(courses, course_df["Credits"].astype(float)))
g_min = dict(zip(courses, course_df["MinGroups"].astype(int)))
g_max = dict(zip(courses, course_df["MaxGroups"].astype(int)))

pref = {}
for _, row in course_df.iterrows():
    c = row["Course"]
    for f in faculties:
        pref[(c, f)] = int(row[f])

L = {faculties[i]: int(min_load_row[i]) for i in range(len(faculties))}
U = {faculties[i]: int(max_load_row[i]) for i in range(len(faculties))}

print("Number of courses:", len(courses))
print("Number of faculties:", len(faculties))
print("Total minimum demand (in groups):", sum(g_min[c] for c in courses))
print("Total maximum demand (in groups):", sum(g_max[c] for c in courses))


Number of courses: 42
Number of faculties: 12
Total minimum demand (in groups): 96
Total maximum demand (in groups): 108


In [9]:
# Quick check of faculty load bounds
load_table = pd.DataFrame({
    "Faculty": faculties,
    "MinLoad": [L[f] for f in faculties],
    "MaxLoad": [U[f] for f in faculties]
})
load_table


,Faculty,MinLoad,MaxLoad
0,Faculty 1,14,15
1,Faculty 2,12,13
2,Faculty 3,10,10
3,Faculty 4,11,16
4,Faculty 5,11,11
5,Faculty 6,11,14
6,Faculty 7,12,14
7,Faculty 8,11,12
8,Faculty 9,12,14
9,Faculty 10,11,11


## Monte Carlo demand scenarios

For each course and each scenario, the realized number of groups is sampled as an integer between:
- `MinGroups`
- `MaxGroups`

This is the uncertainty handled in stage 2.


In [10]:
# =========================
# 3. Generate Monte Carlo scenarios
# =========================
np.random.seed(42)

S = 100   # number of scenarios
scenarios = list(range(S))

g_s = {}
for s in scenarios:
    for c in courses:
        g_s[(c, s)] = np.random.randint(g_min[c], g_max[c] + 1)

# Example: first 5 scenario values for first 5 courses
example_rows = []
for c in courses[:5]:
    row = {"Course": c}
    for s in scenarios[:5]:
        row[f"Scenario_{s}"] = g_s[(c, s)]
    example_rows.append(row)

pd.DataFrame(example_rows)


,Course,Scenario_0,Scenario_1,Scenario_2,Scenario_3,Scenario_4
0,T101,3,3,3,3,3
1,T102,3,3,3,3,3
2,T103,2,2,2,2,2
3,T104,3,3,4,4,4
4,T105,2,1,2,1,2


## Parameter settings

You can adjust these values later.

- `adjunct_cost_stage1`: planned adjunct cost per group
- `adjunct_cost_stage2`: extra adjunct recourse cost per group
- `lambda_pref`: penalty weight for low-preference assignments

A higher `lambda_pref` means the model cares more about teacher-course matching.


In [11]:
# =========================
# 4. Hyperparameters
# =========================
adjunct_cost_stage1 = 10
adjunct_cost_stage2 = 12
lambda_pref = 1

print("Stage 1 adjunct cost:", adjunct_cost_stage1)
print("Stage 2 adjunct cost:", adjunct_cost_stage2)
print("Preference penalty weight:", lambda_pref)


Stage 1 adjunct cost: 10
Stage 2 adjunct cost: 12
Preference penalty weight: 1


## Build the PuLP model

### Decision variables
- `y[c,f]`: integer number of groups of course `c` assigned to full-time faculty `f` in stage 1
- `a[c]`: integer number of groups of course `c` assigned to adjuncts in stage 1
- `u[c,s]`: integer extra adjunct groups used in stage 2 under scenario `s`

### Objective
Minimize:
1. stage-1 adjunct cost  
2. preference mismatch penalty  
3. expected stage-2 recourse cost


In [12]:
# =========================
# 5. Build the optimization model
# =========================
model = pulp.LpProblem("TwoStageTeachingAssignment", pulp.LpMinimize)

# Stage 1 integer variables
y = pulp.LpVariable.dicts(
    "y",
    ((c, f) for c in courses for f in faculties),
    lowBound=0,
    cat="Integer"
)

a = pulp.LpVariable.dicts(
    "a",
    (c for c in courses),
    lowBound=0,
    cat="Integer"
)

# Stage 2 integer recourse variables
u = pulp.LpVariable.dicts(
    "u",
    ((c, s) for c in courses for s in scenarios),
    lowBound=0,
    cat="Integer"
)

# Objective
stage1_adjunct_cost = pulp.lpSum(adjunct_cost_stage1 * a[c] for c in courses)

preference_penalty = pulp.lpSum(
    lambda_pref * (5 - pref[(c, f)]) * y[(c, f)]
    for c in courses for f in faculties
)

stage2_recourse_cost = (1 / S) * pulp.lpSum(
    adjunct_cost_stage2 * u[(c, s)]
    for c in courses for s in scenarios
)

model += stage1_adjunct_cost + preference_penalty + stage2_recourse_cost


In [13]:
# =========================
# 6. Add constraints
# =========================

# (1) Stage 1 must cover baseline demand = MinGroups
for c in courses:
    model += (
        pulp.lpSum(y[(c, f)] for f in faculties) + a[c] == g_min[c],
        f"baseline_cover_{c}"
    )

# (2) Every full-time faculty must be used within min/max teaching load
for f in faculties:
    model += (
        pulp.lpSum(credits[c] * y[(c, f)] for c in courses) >= L[f],
        f"min_load_{f}"
    )
    model += (
        pulp.lpSum(credits[c] * y[(c, f)] for c in courses) <= U[f],
        f"max_load_{f}"
    )

# (3) Stage 2 recourse: extra adjunct if realized demand exceeds stage-1 plan
for s in scenarios:
    for c in courses:
        model += (
            pulp.lpSum(y[(c, f)] for f in faculties) + a[c] + u[(c, s)] >= g_s[(c, s)],
            f"recourse_{c}_{s}"
        )

print("Number of variables:", len(model.variables()))
print("Number of constraints:", len(model.constraints))


Number of variables: 4746
Number of constraints: 4266


In [14]:
# =========================
# 7. Solve
# =========================
solver = pulp.PULP_CBC_CMD(msg=True)
result_status = model.solve(solver)

print("Solver status:", pulp.LpStatus[result_status])
print("Objective value:", pulp.value(model.objective))


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.13/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/7t/qqdmcxf14sj61zyqs87mx6r00000gn/T/fcda7d0448ce4ff883d0eb86f2475fb1-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/7t/qqdmcxf14sj61zyqs87mx6r00000gn/T/fcda7d0448ce4ff883d0eb86f2475fb1-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 4271 COLUMNS
At line 78782 RHS
At line 83049 BOUNDS
At line 87796 ENDATA
Problem MODEL has 4266 rows, 4746 columns and 60354 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 297.713 - 0.00 seconds
Cgl0003I 0 fixed, 4200 tightened bounds, 0 strengthened rows, 0 substitutions
Cgl0004I processed model has 4254 rows, 4746 columns (4746 integer (130 of which binary)) and 59850 elements
Cutoff increment increased from 1e-05 to 0.03996
Cbc0012I

In [15]:
# =========================
# 8. Extract main results
# =========================
if pulp.LpStatus[result_status] != "Optimal":
    print("No optimal solution found.")
else:
    # Stage 1 adjunct assignment
    adjunct_stage1_df = pd.DataFrame({
        "Course": courses,
        "Stage1_Adjunct_Groups": [int(pulp.value(a[c])) for c in courses]
    })
    adjunct_stage1_df = adjunct_stage1_df[adjunct_stage1_df["Stage1_Adjunct_Groups"] > 0].reset_index(drop=True)

    # Faculty assignment
    faculty_rows = []
    for c in courses:
        for f in faculties:
            val = int(pulp.value(y[(c, f)]))
            if val > 0:
                faculty_rows.append({
                    "Course": c,
                    "Faculty": f,
                    "Assigned_Groups": val,
                    "Preference": pref[(c, f)],
                    "Credits_per_Group": credits[c]
                })
    faculty_assign_df = pd.DataFrame(faculty_rows)

    # Average stage 2 extra adjunct
    extra_adjunct_summary = []
    for c in courses:
        avg_extra = np.mean([pulp.value(u[(c, s)]) for s in scenarios])
        max_extra = np.max([pulp.value(u[(c, s)]) for s in scenarios])
        extra_adjunct_summary.append({
            "Course": c,
            "Avg_Extra_Adjunct_Groups": round(avg_extra, 2),
            "Max_Extra_Adjunct_Groups": int(max_extra)
        })
    extra_adjunct_df = pd.DataFrame(extra_adjunct_summary)

    print("\nStage 1 adjunct assignments:")
    display(adjunct_stage1_df.head(20))

    print("\nFaculty teaching assignments:")
    display(faculty_assign_df.head(30))

    print("\nStage 2 extra adjunct summary:")
    display(extra_adjunct_df.head(20))



Stage 1 adjunct assignments:


,Course,Stage1_Adjunct_Groups
0,T101,3
1,T106,3
2,T111,1
3,T202,3
4,T203,1
5,T204,2
6,T207,1
7,T208,2
8,T209,1
9,T213,2



Faculty teaching assignments:


,Course,Faculty,Assigned_Groups,Preference,Credits_per_Group
0,T102,Faculty 5,1,5,2.0
1,T102,Faculty 7,1,4,2.0
2,T102,Faculty 10,1,5,2.0
3,T103,Faculty 12,2,5,2.0
4,T104,Faculty 8,3,5,2.0
5,T105,Faculty 10,1,5,2.0
6,T107,Faculty 1,1,5,2.0
7,T107,Faculty 11,2,5,2.0
8,T108,Faculty 4,2,5,2.0
9,T109,Faculty 11,1,5,2.0



Stage 2 extra adjunct summary:


,Course,Avg_Extra_Adjunct_Groups,Max_Extra_Adjunct_Groups
0,T101,0.00,0
1,T102,0.00,0
2,T103,0.00,0
3,T104,0.49,1
4,T105,0.53,1
5,T106,0.00,0
6,T107,0.46,1
7,T108,0.00,0
8,T109,0.00,0
9,T110,0.00,0


In [16]:
# =========================
# 9. Check faculty workloads
# =========================
workload_rows = []
for f in faculties:
    assigned_credits = sum(credits[c] * pulp.value(y[(c, f)]) for c in courses)
    workload_rows.append({
        "Faculty": f,
        "MinLoad": L[f],
        "AssignedCredits": assigned_credits,
        "MaxLoad": U[f]
    })

workload_df = pd.DataFrame(workload_rows)
workload_df


,Faculty,MinLoad,AssignedCredits,MaxLoad
0,Faculty 1,14,15.0,15
1,Faculty 2,12,13.0,13
2,Faculty 3,10,10.0,10
3,Faculty 4,11,16.0,16
4,Faculty 5,11,11.0,11
5,Faculty 6,11,14.0,14
6,Faculty 7,12,14.0,14
7,Faculty 8,11,12.0,12
8,Faculty 9,12,14.0,14
9,Faculty 10,11,11.0,11


In [17]:
# =========================
# 10. Save outputs to Excel (optional)
# =========================
output_file = "two_stage_teaching_results_pulp.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    adjunct_stage1_df.to_excel(writer, sheet_name="Stage1_Adjunct", index=False)
    faculty_assign_df.to_excel(writer, sheet_name="Faculty_Assignment", index=False)
    extra_adjunct_df.to_excel(writer, sheet_name="Stage2_Extra_Adjunct", index=False)
    workload_df.to_excel(writer, sheet_name="Faculty_Workload", index=False)

print("Saved results to:", output_file)


Saved results to: two_stage_teaching_results_pulp.xlsx
